# Player Behavior Classification - Exploratory Analysis

This notebook covers:
1. Synthetic data generation and inspection
2. Exploratory data analysis (EDA)
3. Feature engineering
4. Model training and evaluation
5. Feature importance analysis

In [ ]:
# Imports and Setup
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Load config
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

RANDOM_SEED = config['random_seed']
np.random.seed(RANDOM_SEED)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Config loaded successfully")
print(f"Random seed: {RANDOM_SEED}")

## 1. Data Generation & Inspection

In [ ]:
# Generate synthetic data
from data_generation import generate_synthetic_telemetry

df = generate_synthetic_telemetry(config)
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:\n{df['behavior_class'].value_counts()}")
print(f"\nFeatures: {list(df.columns.drop('behavior_class'))}")

In [ ]:
# Basic statistics
print(df.describe())

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

## 2. Exploratory Data Analysis

In [ ]:
# Feature distributions by class
features = [c for c in df.columns if c != 'behavior_class']

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    for cls in df['behavior_class'].unique():
        subset = df[df['behavior_class'] == cls][feat]
        axes[i].hist(subset, alpha=0.5, label=cls, bins=30, density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions_by_class.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Pairplot for key features (subset for readability)
key_features = features[:6]  # First 6 features
sns.pairplot(df[key_features + ['behavior_class']], hue='behavior_class', diag_kind='kde', plot_kws={'alpha': 0.5})
plt.savefig('../reports/figures/pairplot_key_features.png', dpi=150)
plt.show()

## 3. Feature Engineering & Train/Val/Test Split

In [ ]:
from features import prepare_features_and_split

X_train, X_val, X_test, y_train, y_val, y_test, label_encoder, scaler = prepare_features_and_split(df, config)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Classes: {label_encoder.classes_}")

## 4. Model Training

In [ ]:
from models import train_all_models

models, results = train_all_models(X_train, y_train, X_val, y_val, config)

for name, metrics in results.items():
    print(f"{name}: Val Accuracy = {metrics['accuracy']:.4f}, F1 = {metrics['f1']:.4f}")

## 5. Evaluation on Test Set

In [ ]:
from evaluation import evaluate_models, plot_confusion_matrices, plot_feature_importance

test_results = evaluate_models(models, X_test, y_test)

for name, metrics in test_results.items():
    print(f"{name}: Test Accuracy = {metrics['accuracy']:.4f}, F1 = {metrics['f1']:.4f}")

In [ ]:
plot_confusion_matrices(models, X_test, y_test, config)

## 6. Feature Importance Analysis

In [ ]:
plot_feature_importance(models, X_test, y_test, config)

## 7. Cross-Validation Results

In [ ]:
from evaluation import cross_validate_models

cv_results = cross_validate_models(models, X_train, y_train, config)

for name, scores in cv_results.items():
    print(f"{name}: CV Accuracy = {scores['accuracy'].mean():.4f} (+/- {scores['accuracy'].std():.4f})")